# 02 - Encoding (Bag of Words)

Este notebook toma los parquets preprocesados y construye la codificacion **BoW** (Bag of Words).

## Objetivo
- Convertir `text_clean` en una matriz numerica sparse BoW.
- Evitar **data leakage**: el vocabulario se aprende solo con train.
- Crear un conjunto de **validacion** desde train para ajustar hiperparametros sin tocar test.

## Politica de conjuntos
| Conjunto | Tamano | Uso |
|---|---|---|
| train | 1,360,000 | Entrenar y crear val |
| val | 20% del train | Ajuste de hiperparametros |
| test | 240,000 | Evaluacion final unica en 03_baseline.ipynb |

## Reproducibilidad
Se fija `SEED = 42` para que todos los experimentos sean replicables.

## 1. Instalacion e imports

In [1]:
!pip install -q scikit-learn pyarrow joblib scipy

In [2]:
# ══════════════════════════════════════════════
EXPERIMENT = "elongacion"   # ← debe coincidir con 01
# ══════════════════════════════════════════════

import pandas as pd
import numpy as np
import joblib
from scipy.sparse import save_npz, load_npz
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

SEED = 42
print('OK - Imports listos. SEED:', SEED)

OK - Imports listos. SEED: 42


## 2. Carga de parquets preprocesados

Cargamos los archivos generados en `01_preprocessing.ipynb`.

Cada parquet debe tener exactamente estas columnas:
- `text_normalized`: texto limpio y lematizado.
- `label`: 0 = negativo, 1 = positivo.

In [3]:
df_train = pd.read_parquet(f'sentiment_prep_train_{EXPERIMENT}.parquet')
df_test  = pd.read_parquet(f'sentiment_prep_test_{EXPERIMENT}.parquet')

print(f"Experimento : {EXPERIMENT}")
print(f"Train shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")
print(df_train.head(3))

Experimento : elongacion
Train shape : (1360000, 2)
Test shape  : (240000, 2)
                                          text_clean  label
0  Closet organizer install complete. Now for the...      0
1  Mornin' All!! ..I need to wake up..this will g...      1
2                       ahh! he suxx! he claimed it!      0


### 2.1 Validaciones previas

Antes de vectorizar verificamos nulos, columnas requeridas y textos vacios.
Los textos vacios ocurren cuando un tweet era solo URL, menciones o hashtags y quedo sin contenido tras el preprocesamiento.

In [4]:
# REVISAR NOMBRE TEXT_CLEAN Y AJUSTAR POR EL PROPIO ANTES DE CUALQUIER CAMBIO

# Verificar nulos y textos vacíos antes de vectorizar
required_cols = {'text_clean', 'label'}
assert required_cols.issubset(df_train.columns), 'Train: faltan columnas requeridas'
assert required_cols.issubset(df_test.columns),  'Test: faltan columnas requeridas'

# Reemplazar posibles NaN en text_normalized por string vacío
df_train['text_clean'] = df_train['text_clean'].fillna('')
df_test['text_clean']  = df_test['text_clean'].fillna('')

print('Nulos train:', df_train.isnull().sum().to_dict())
print('Nulos test: ', df_test.isnull().sum().to_dict())
print('Textos vacios train:', (df_train['text_clean'] == '').sum())
print('Textos vacios test: ', (df_test['text_clean']  == '').sum())
print('Balance labels train:', df_train['label'].value_counts().to_dict())
print('Balance labels test: ', df_test['label'].value_counts().to_dict())

Nulos train: {'text_clean': 0, 'label': 0}
Nulos test:  {'text_clean': 0, 'label': 0}
Textos vacios train: 2634
Textos vacios test:  474
Balance labels train: {1: 680129, 0: 679871}
Balance labels test:  {0: 120129, 1: 119871}


## 3. Bag of Words con CountVectorizer

### Que es BoW?
BoW convierte cada tweet en un vector donde cada posicion representa una palabra del vocabulario y el valor es cuantas veces aparece. El orden de las palabras se pierde.

### Regla critica: evitar data leakage
- `fit_transform(train)`: aprende el vocabulario **solo** con train.
- `transform(test)`: aplica el vocabulario aprendido, no aprende nada nuevo.

Si hicieras `fit` con train+test, el vocabulario incluiria palabras del test y las metricas quedarian infladas artificialmente.

### Parametros del vectorizador
| Parametro | Valor | Significado |
|---|---|---|
| `max_features` | 50,000 | Solo las 50k palabras mas frecuentes |
| `min_df` | 5 | Ignorar palabras en menos de 5 tweets |
| `max_df` | 0.95 | Ignorar palabras en mas del 95% de tweets |
| `ngram_range` | (1,1) | Solo unigramas |

In [5]:
# Separar features y labels
X_raw_train = df_train['text_clean'].values
y_train      = df_train['label'].values

X_raw_test  = df_test['text_clean'].values
y_test       = df_test['label'].values

print(f"Textos en train: {len(X_raw_train):,}")
print(f"Textos en test:  {len(X_raw_test):,}")
print(f"Distribución labels train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Distribución labels test:  {dict(zip(*np.unique(y_test,  return_counts=True)))}")

Textos en train: 1,360,000
Textos en test:  240,000
Distribución labels train: {np.int64(0): np.int64(679871), np.int64(1): np.int64(680129)}
Distribución labels test:  {np.int64(0): np.int64(120129), np.int64(1): np.int64(119871)}


In [6]:
# Definir y ajustar el vectorizador SOLO sobre train
# (nunca hacer fit sobre test — data leakage)
vectorizer = CountVectorizer(
    max_features=50000,   # Top 50k palabras más frecuentes
    min_df=5,             # Ignorar palabras en menos de 5 documentos
    max_df=0.95,          # Ignorar palabras en más del 95% de documentos
    ngram_range=(1, 1)    # Unigramas (una palabra a la vez)
)

# fit_transform sobre train — aprende vocabulario y transforma
X_train_bow = vectorizer.fit_transform(X_raw_train)

# transform sobre test — aplica el vocabulario aprendido del train
X_test_bow  = vectorizer.transform(X_raw_test)

print(f"Shape X_train_bow: {X_train_bow.shape}")
print(f"Shape X_test_bow:  {X_test_bow.shape}")
print(f"Tipo de matriz:    {type(X_train_bow)}")
print(f"Vocabulario final: {len(vectorizer.vocabulary_):,} palabras")

Shape X_train_bow: (1360000, 46635)
Shape X_test_bow:  (240000, 46635)
Tipo de matriz:    <class 'scipy.sparse._csr.csr_matrix'>
Vocabulario final: 46,635 palabras


## 4. Diagnostico del BoW

Analizamos las palabras mas frecuentes para verificar que la limpieza fue correcta, y calculamos la **sparsity** de la matriz.

La sparsity mide el porcentaje de ceros. Es normal que sea mayor al 99% porque cada tweet usa solo una fraccion del vocabulario total de 50k palabras.

In [7]:
# Top 20 palabras más frecuentes en el corpus
freq  = X_train_bow.sum(axis=0).A1
words = vectorizer.get_feature_names_out()
top20 = pd.Series(freq, index=words).sort_values(ascending=False).head(20)
print('Top 20 palabras mas frecuentes:\n')
print(top20.to_string())

Top 20 palabras mas frecuentes:

to      481702
the     445225
my      269187
it      258954
and     257718
you     256661
is      200936
in      184221
for     183914
of      156051
on      143345
that    141033
me      138739
so      131281
have    123697
but     113333
just    107972
with     98171
be       95983
at       95459


In [8]:
# Calcular sparsity de la matriz
total_elementos  = X_train_bow.shape[0] * X_train_bow.shape[1]
elementos_nonzero = X_train_bow.nnz
sparsity = (1 - elementos_nonzero / total_elementos) * 100

print(f"Total elementos posibles: {total_elementos:,}")
print(f"Elementos no-cero:        {elementos_nonzero:,}")
print(f"Sparsity:                 {sparsity:.2f}%")

Total elementos posibles: 63,423,600,000
Elementos no-cero:        14,903,200
Sparsity:                 99.98%


## 5. Split de validacion (desde train)

Ya tenemos train y test separados desde los parquets. Sin embargo, **test queda reservado para la evaluacion final** y no se puede tocar durante el desarrollo.

Para poder experimentar con hiperparametros en `03_baseline.ipynb` necesitamos un tercer conjunto: **validacion**.

Dividimos `X_train_bow` en:
- `X_tr` (80%): para entrenar los modelos durante el desarrollo.
- `X_val` (20%): para comparar modelos y ajustar hiperparametros.

Una vez elegidos los hiperparametros en `03_baseline.ipynb`, el modelo final se re-entrena con `X_train_bow` completo y se evalua **una sola vez** con `X_test_bow`.

> `stratify=y_train` garantiza que X_tr y X_val mantengan el balance 50/50 de labels.

In [9]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_bow,
    y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train
)

print('X_tr  (80% train):', X_tr.shape)
print('X_val (20% train):', X_val.shape)
print('X_test (intocado):', X_test_bow.shape)
print()
print('Balance y_tr :', dict(zip(*np.unique(y_tr,  return_counts=True))))
print('Balance y_val:', dict(zip(*np.unique(y_val, return_counts=True))))

X_tr  (80% train): (1088000, 46635)
X_val (20% train): (272000, 46635)
X_test (intocado): (240000, 46635)

Balance y_tr : {np.int64(0): np.int64(543897), np.int64(1): np.int64(544103)}
Balance y_val: {np.int64(0): np.int64(135974), np.int64(1): np.int64(136026)}


## 6. Guardar artefactos

Guardamos todos los objetos que necesita `03_baseline.ipynb`:

| Archivo | Descripcion |
|---|---|
| `X_tr.npz` | Train 80% para entrenar durante desarrollo |
| `X_val.npz` | Validacion 20% para ajustar hiperparametros |
| `X_train_bow.npz` | Train completo para entrenamiento final del modelo |
| `X_test_bow.npz` | Test reservado para evaluacion final unica |
| `y_tr/val/train/test.pkl` | Labels correspondientes a cada split |
| `bow_vectorizer.pkl` | Vectorizador ajustado (necesario para inferencia futura) |

In [10]:
save_npz(f'X_tr_{EXPERIMENT}.npz',        X_tr)
save_npz(f'X_val_{EXPERIMENT}.npz',       X_val)
save_npz(f'X_train_bow_{EXPERIMENT}.npz', X_train_bow)
save_npz(f'X_test_bow_{EXPERIMENT}.npz',  X_test_bow)

joblib.dump(y_tr,    f'y_tr_{EXPERIMENT}.pkl')
joblib.dump(y_val,   f'y_val_{EXPERIMENT}.pkl')
joblib.dump(y_train, f'y_train_{EXPERIMENT}.pkl')
joblib.dump(y_test,  f'y_test_{EXPERIMENT}.pkl')
joblib.dump(vectorizer, f'bow_vectorizer_{EXPERIMENT}.pkl')

print(f"✅ Artefactos guardados para: {EXPERIMENT}")

✅ Artefactos guardados para: elongacion


## 7. Verificacion final

Cargamos los archivos guardados para confirmar shapes correctas antes de pasar a `03_baseline.ipynb`.

In [11]:
print('Verificacion de artefactos guardados:')
print('  X_tr:        ', load_npz(f'X_tr_{EXPERIMENT}.npz').shape)
print('  X_val:       ', load_npz(f'X_val_{EXPERIMENT}.npz').shape)
print('  X_train_bow: ', load_npz(f'X_train_bow_{EXPERIMENT}.npz').shape)
print('  X_test_bow:  ', load_npz(f'X_test_bow_{EXPERIMENT}.npz').shape)
print(f'\n✅ Todos los artefactos del experimento [{EXPERIMENT}] guardados correctamente')


Verificacion de artefactos guardados:
  X_tr:         (1088000, 46635)
  X_val:        (272000, 46635)
  X_train_bow:  (1360000, 46635)
  X_test_bow:   (240000, 46635)

✅ Todos los artefactos del experimento [elongacion] guardados correctamente
